In [ ]:
# Install dependencies if you haven't already:
# Uninstall the CPU versions to prevent conflicts
# %pip uninstall -y torch torchvision torchaudio

# Install the GPU (CUDA) versions directly from PyTorch
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Install the rest of the required libraries
# %pip install ultralytics opencv-python scikit-learn matplotlib numpy

import cv2
import glob
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# 1. Configuration
DATA_DIR = "data/images" # Ensure you have data/images/squat_standing and data/images/squat_bottom
TEST_VIDEO_PATH = "test_video.mp4" # Update with your test video
OUTPUT_VIDEO_PATH = "rep_count_output.mp4"
POSE_CONFIDENCE = 0.5

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load YOLOv8 Nano Pose (downloads automatically if missing)
print("Loading YOLOv8 Pose model...")
yolo_pose_model = YOLO('yolov8n-pose.pt')

In [ ]:
def process_image_dataset(data_dir, yolo_model):
    #  Your_Project_Folder/              
    # ├── Rep_Count.ipynb      
    # ├── test_video.mp4                
    # └── data/
    #   └── images/
    #       ├── squat_standing/       
    #       └── squat_bottom/          
    X_data = []
    y_data = []
    classes = ['squat_standing', 'squat_bottom']
    
    for class_name in classes:
        # Check for multiple common image extensions
        image_paths = glob.glob(f"{data_dir}/{class_name}/*.*")
        print(f"Found {len(image_paths)} images for {class_name}")
        
        for img_path in image_paths:
            frame = cv2.imread(img_path)
            if frame is None: 
                continue
                
            # Run YOLO inference
            results = yolo_model(frame, verbose=False)
            
            if len(results) > 0 and results[0].keypoints is not None:
                kpts_data = results[0].keypoints.data
                if len(kpts_data) > 0:
                    kpts = kpts_data[0].cpu().numpy() # Shape: (17, 3)
                    
                    # Only use high-confidence detections
                    if np.mean(kpts[:, 2]) >= POSE_CONFIDENCE:
                        # Normalize coordinates so the model is resolution-independent
                        h, w = frame.shape[:2]
                        kpts_normalized = kpts.copy()
                        kpts_normalized[:, 0] /= w
                        kpts_normalized[:, 1] /= h
                        
                        X_data.append(kpts_normalized.flatten())
                        y_data.append(class_name)
                        
    return np.array(X_data, dtype=np.float32), np.array(y_data)

# Process the dataset
print("Extracting keypoints from images...")
X_raw, y_raw = process_image_dataset(DATA_DIR, yolo_pose_model)

if len(X_raw) == 0:
    raise ValueError("No valid keypoints found. Check your image paths and dataset folders.")

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)

# Split into Train and Validation sets
X_train, X_val, y_train, y_val = train_test_split(X_raw, y_encoded, test_size=0.2, random_state=42)

# Convert to PyTorch Dataloaders
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Dataset ready: {len(X_train)} train samples, {len(X_val)} validation samples.")

In [ ]:
class RepCounterModel(nn.Module):
    def __init__(self, input_size=51, hidden_sizes=[128, 64], num_classes=2):
        super(RepCounterModel, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_size, hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_sizes[1], num_classes)
        )
    
    def forward(self, keypoints):
        return self.classifier(keypoints)

# Initialize Model, Loss, and Optimizer
model = RepCounterModel(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training Loop
epochs = 220
print("Starting training...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == batch_y).sum().item()
        
    train_acc = 100 * correct / len(train_dataset)
    
    # Simple validation log every 10 epochs
    if (epoch + 1) % 10 == 0:
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for val_X, val_y in val_loader:
                val_X, val_y = val_X.to(device), val_y.to(device)
                val_outputs = model(val_X)
                _, val_predicted = torch.max(val_outputs.data, 1)
                val_correct += (val_predicted == val_y).sum().item()
        val_acc = 100 * val_correct / len(val_dataset)
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {total_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("Training complete!")

In [ ]:
def count_reps_in_video(video_path, output_path, custom_model, yolo_model, encoder):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    custom_model.eval()
    
    # State Machine Variables
    rep_count = 0
    is_in_bottom_pose = False
    confidence_threshold = 0.75 # Must be highly confident to switch states
    
    print("Processing video...")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        annotated_frame = frame.copy()
        
        # 1. Get Skeletal Data
        results = yolo_model(frame, verbose=False)
        
        if len(results) > 0 and results[0].keypoints is not None:
            kpts_data = results[0].keypoints.data
            if len(kpts_data) > 0:
                kpts = kpts_data[0].cpu().numpy()
                
                if np.mean(kpts[:, 2]) >= POSE_CONFIDENCE:
                    # Normalize
                    kpts_normalized = kpts.copy()
                    kpts_normalized[:, 0] /= width
                    kpts_normalized[:, 1] /= height
                    
                    keypoints_tensor = torch.tensor(kpts_normalized.flatten(), dtype=torch.float32).unsqueeze(0).to(device)
                    
                    # 2. Predict Pose State
                    with torch.no_grad():
                        logits = custom_model(keypoints_tensor)
                        probs = torch.nn.functional.softmax(logits, dim=1)[0]
                        predicted_idx = torch.argmax(probs).item()
                        confidence = probs[predicted_idx].item()
                        predicted_class = encoder.classes_[predicted_idx]
                    
                    color = (0, 255, 0) # Default Green
                    
                    # 3. State Machine Logic
                    if confidence >= confidence_threshold:
                        if predicted_class == 'squat_bottom':
                            is_in_bottom_pose = True
                            color = (0, 165, 255) # Orange (going down)
                            
                        elif predicted_class == 'squat_standing' and is_in_bottom_pose:
                            rep_count += 1
                            is_in_bottom_pose = False # Reset flag
                            color = (0, 255, 0) # Green (completed)
                    
                    # 4. Draw overlays
                    # Draw a semi-transparent background box for text
                    cv2.rectangle(annotated_frame, (10, 10), (450, 120), (0, 0, 0), -1)
                    cv2.addWeighted(annotated_frame, 0.6, frame, 0.4, 0, annotated_frame)
                    
                    cv2.putText(annotated_frame, f"REPS: {rep_count}", (20, 60),
                                cv2.FONT_HERSHEY_DUPLEX, 1.5, (0, 255, 255), 3)
                    cv2.putText(annotated_frame, f"State: {predicted_class} ({confidence:.2f})", (20, 100),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        out.write(annotated_frame)

    cap.release()
    out.release()
    print(f"Video saved to {output_path}")

# Run the inference
count_reps_in_video(TEST_VIDEO_PATH, OUTPUT_VIDEO_PATH, model, yolo_pose_model, label_encoder)

In [ ]:
%pip install tensorflow==2.16.1 tf-keras

%pip install ml-dtypes==0.3.2


In [ ]:
import torch
import tensorflow as tf
import numpy as np
import os

# FORCE TENSORFLOW TO USE LEGACY KERAS
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tf_keras as keras

print("Step 1: Recreating architecture in Legacy Keras...")
# Use keras (tf_keras) instead of tf.keras
keras_model = keras.Sequential([
    keras.layers.Input(shape=(51,)), 
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(2)
])

print("Step 2: Transferring trained PyTorch weights...")
pt_weights = model.state_dict()

# PyTorch to Keras weight mapping
w0 = pt_weights['classifier.0.weight'].cpu().numpy().T
b0 = pt_weights['classifier.0.bias'].cpu().numpy()
w3 = pt_weights['classifier.3.weight'].cpu().numpy().T
b3 = pt_weights['classifier.3.bias'].cpu().numpy()
w6 = pt_weights['classifier.6.weight'].cpu().numpy().T
b6 = pt_weights['classifier.6.bias'].cpu().numpy()

keras_model.layers[0].set_weights([w0, b0])
keras_model.layers[1].set_weights([w3, b3])
keras_model.layers[2].set_weights([w6, b6])

print("Step 3: Quantizing to INT8...")
# Pass the keras_model explicitly to the TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    for i in range(min(200, len(X_train))):
        sample = X_train[i].astype(np.float32)
        yield [np.expand_dims(sample, axis=0)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# This should now execute without the NoneType error
tflite_model_int8 = converter.convert()

In [ ]:
# Save the quantized model to a file
tflite_model_path = "rep_counter_int8.tflite"
with open(tflite_model_path, "wb") as f:
    f.write(tflite_model_int8)

print(f"Successfully saved quantized model to: {tflite_model_path}")